In [ ]:
import sys, shutil
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
OUT_DIR = Path('/kaggle/working')

print('Contents of /kaggle/input:', sorted(p.name for p in INPUT_ROOT.iterdir()))


_SKIP_DIR_NAMES = {'train_series', 'test_series'}


def find_dir_containing(marker_relpath, root=INPUT_ROOT, max_depth=8):
    """Breadth-first search under /kaggle/input for a directory containing
    `marker_relpath`, instead of assuming a fixed mount layout. Confirmed
    inconsistent across source types: competitions nest under 'competitions/<slug>/',
    some datasets mount flat at the top level, others nest under
    'datasets/<owner>/<name>/', and Model sources nest deepest (depth 5-6).

    Prunes train_series/test_series before descending into them -- same lesson as
    the public baseline notebook's cover-image loader: the competition mount holds
    hundreds of thousands of per-study/per-series directories, and none of our
    markers (package init files, config.json, checkpoint files) live inside them.
    A naive glob-at-each-depth search (the first version of this function) doesn't
    prune, so it ends up enumerating that entire tree once search depth reaches it
    -- confirmed costing ~8 minutes per kernel run before this fix.
    """
    frontier = [root]
    for _ in range(max_depth + 1):
        next_frontier = []
        for candidate in frontier:
            if (candidate / marker_relpath).exists():
                return candidate
            try:
                children = [c for c in candidate.iterdir() if c.is_dir() and c.name not in _SKIP_DIR_NAMES]
            except OSError:
                continue
            next_frontier.extend(children)
        frontier = next_frontier
    raise FileNotFoundError(
        f'Could not find a directory containing {marker_relpath!r} under {root} '
        f'(searched up to depth {max_depth}). Top-level contents: '
        f'{sorted(p.name for p in root.iterdir())}'
    )


SRC_DATASET = find_dir_containing('rsna_knee/__init__.py')
print('Resolved SRC_DATASET:', SRC_DATASET)
sys.path.insert(0, str(SRC_DATASET))

COMPETITION_ROOT = find_dir_containing('sample_submission.csv')
print('Resolved COMPETITION_ROOT:', COMPETITION_ROOT)
print('Its contents:', sorted(p.name for p in COMPETITION_ROOT.iterdir()))

# preprocessor_config.json is specific to HuggingFace vision-model checkpoints (e.g.
# the mounted DINOv2 Model source) -- unambiguous among our attached inputs.
DINOV2_PATH = find_dir_containing('preprocessor_config.json')
print('Resolved DINOV2_PATH:', DINOV2_PATH)

# Write a trivial valid submission first -- if anything below fails partway, this
# is what gets scored instead of nothing.
shutil.copy(COMPETITION_ROOT / 'sample_submission.csv', OUT_DIR / 'submission.csv')
print('Trivial fallback submission written.')

In [ ]:
import numpy as np
import pandas as pd
import torch

from rsna_knee.data.series import assign_slots
from rsna_knee.data.label_expansion import add_derived_confidence, build_training_targets
from rsna_knee.data.labels import load_train_labels
from rsna_knee.datasets import StudyDataset
from rsna_knee.metrics import LABEL_COLUMNS, per_label_auc
from rsna_knee.models.vision import KneeMultiLabelModel
from rsna_knee.train import train_one_epoch
from rsna_knee.infer import run_inference
from rsna_knee.validate import report_hash, select_holdout_ids

def select_device():
    """torch.cuda.is_available() only checks that a CUDA driver/device is visible,
    not that the installed PyTorch build actually supports its compute capability --
    Kaggle sometimes assigns an older GPU (e.g. a P100, sm_60) that a newer PyTorch
    wheel has dropped kernel support for, which fails on the first real op rather
    than at is_available() time. Do a tiny real op and fall back to CPU if it fails.
    """
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        (torch.zeros(1, device='cuda') + 1).cpu()
        return 'cuda'
    except Exception as e:
        print(f'CUDA reported available but failed a sanity op ({e!r}); falling back to CPU.')
        return 'cpu'


device = select_device()
print('device:', device)

In [ ]:
BATCH_SIZE = 16
EPOCHS = 3
N_UNFROZEN_BLOCKS = 2
ENCODER_LR = 1e-5
HEAD_LR = 1e-3

train_labels = load_train_labels(COMPETITION_ROOT / 'train.csv')
train_series_csv = pd.read_csv(COMPETITION_ROOT / 'train_series.csv')

# Silver labels: stevenleehans/rsna-knee-llm-report-labels (llm_labels_v4_blend.csv),
# validated against the 58 gold studies at 0.893 macro AUC -- better than, and not
# leaked unlike some other public candidates (see label_expansion.py docstring for
# the leakage check). No native confidence column, so derive one.
SILVER_ROOT = find_dir_containing('llm_labels_v4_blend.csv')
silver = pd.read_csv(SILVER_ROOT / 'llm_labels_v4_blend.csv')
silver = add_derived_confidence(silver)
targets = build_training_targets(train_labels, silver)

train_labels['report_hash'] = train_labels['Report'].map(report_hash)
train_ids, holdout_ids = select_holdout_ids(
    train_labels, group_col='report_hash', n_splits=5, seed=42, holdout_fold=0
)
print(f'train: {len(train_ids)} studies, holdout: {len(holdout_ids)} studies')

train_series_dir = COMPETITION_ROOT / 'train_series'
train_series_subset = train_series_csv[train_series_csv['StudyInstanceUID'].isin(train_ids)]
train_series_with_slots = assign_slots(train_series_subset, train_series_dir)

train_dataset = StudyDataset(train_ids, train_series_with_slots, train_series_dir, targets)
# persistent_workers=True is required for the dataset's per-study image cache to help
# beyond the first epoch -- PyTorch respawns worker processes every epoch by default,
# which would otherwise silently discard each worker's in-process cache every time.
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, persistent_workers=True
)
print('train dataset size:', len(train_dataset))

In [ ]:
model = KneeMultiLabelModel(
    num_slots=6, num_labels=12, hf_pretrained_path=str(DINOV2_PATH)
).to(device)
model.encoder.freeze_except_last_n_blocks(N_UNFROZEN_BLOCKS)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'trainable params: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)')

optimizer = torch.optim.AdamW(model.discriminative_param_groups(ENCODER_LR, HEAD_LR))

for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, optimizer, device)
    print(f'epoch {epoch+1}/{EPOCHS}: loss={loss:.4f}')

torch.save(model.state_dict(), OUT_DIR / 'model_v2.pt')
print('Checkpoint saved.')

In [ ]:
holdout_series_subset = train_series_csv[train_series_csv['StudyInstanceUID'].isin(holdout_ids)]
holdout_series_with_slots = assign_slots(holdout_series_subset, train_series_dir)

holdout_predictions = run_inference(
    model, holdout_ids, holdout_series_with_slots, train_series_dir, device, batch_size=BATCH_SIZE
)
holdout_predictions = holdout_predictions.set_index('StudyInstanceUID')

# Tier 1: silver-label holdout -- noisy but plenty of samples per class, drives
# model/epoch selection. Silver targets are graded/continuous scores, not binary --
# sklearn's roc_auc_score requires discrete ground truth, so binarize at the
# midpoint first (same approach as the public baseline notebook: "the graded
# targets are binarised at their midpoint so that a held-out AUC can be computed
# at all"). This is purely for computing a local diagnostic AUC -- the actual
# training targets/weights (in `targets`) stay graded and untouched.
holdout_silver_targets = targets[targets['StudyInstanceUID'].isin(holdout_ids)].set_index('StudyInstanceUID')
y_true_silver = (holdout_silver_targets.loc[holdout_predictions.index, LABEL_COLUMNS] >= 0.5).astype(float)
silver_scores = per_label_auc(y_true_silver.reset_index(drop=True), holdout_predictions.reset_index(drop=True))
print('Holdout AUC vs SILVER labels (tier 1, primary signal):')
for col, v in silver_scores.items():
    print(f'  {col:20s} {v:.3f}' if not np.isnan(v) else f'  {col:20s} n/a')
silver_vals = [v for v in silver_scores.values() if not np.isnan(v)]
print(f'  MACRO: {np.mean(silver_vals):.4f}' if silver_vals else '  MACRO: n/a')

# Tier 2: the gold-labeled studies that landed in this holdout -- approximates the
# real competition metric but too few samples to arbitrate alone; report only.
holdout_gold_mask = train_labels.set_index('StudyInstanceUID').loc[holdout_ids, LABEL_COLUMNS].notna().any(axis=1)
holdout_gold_ids = [sid for sid in holdout_ids if holdout_gold_mask.get(sid, False)]
print(f'\nHoldout AUC vs GOLD labels (tier 2, annotation check, n={len(holdout_gold_ids)}):')
if len(holdout_gold_ids) >= 5:
    gold_true = train_labels.set_index('StudyInstanceUID').loc[holdout_gold_ids, LABEL_COLUMNS]
    gold_pred = holdout_predictions.loc[holdout_gold_ids, LABEL_COLUMNS]
    gold_scores = per_label_auc(gold_true.reset_index(drop=True), gold_pred.reset_index(drop=True))
    for col, v in gold_scores.items():
        print(f'  {col:20s} {v:.3f}' if not np.isnan(v) else f'  {col:20s} n/a')
    gold_vals = [v for v in gold_scores.values() if not np.isnan(v)]
    print(f'  MACRO: {np.mean(gold_vals):.4f}' if gold_vals else '  MACRO: n/a')
else:
    print('  too few gold studies in this holdout to report per-label AUC')

In [ ]:
test_csv = pd.read_csv(COMPETITION_ROOT / 'test.csv')
test_series_csv = pd.read_csv(COMPETITION_ROOT / 'test_series.csv')
test_series_dir = COMPETITION_ROOT / 'test_series'

test_study_ids = test_csv['StudyInstanceUID'].tolist()
test_series_with_slots = assign_slots(test_series_csv, test_series_dir)

predictions = run_inference(
    model, test_study_ids, test_series_with_slots, test_series_dir, device, batch_size=BATCH_SIZE
)
predictions.to_csv(OUT_DIR / 'submission.csv', index=False)
print(f'Wrote {len(predictions)} real predictions to submission.csv')
predictions.head()